In [7]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import requests
import dash
from dash import dcc, html, Input, Output, dash_table
import plotly.express as px

class NiftyOptionsDashboard():
    def __init__(self):
        self.app = dash.Dash(__name__)
        self.df = None
        self.expiries = []
        self.layout()
        self.register_callbacks()
        self.fetch_options_chain()

    def fetch_options_chain(self):
        url = "https://www.nseindia.com/api/option-chain-indices?symbol=NIFTY"
        headers = {"User-Agent": "Mozilla/5.0"}
        session = requests.Session()
        session.get("https://www.nseindia.com", headers=headers)

        response = session.get(url, headers=headers)
        data = response.json()
        options = data['records']['data']

        all_options = []
        for option in options:
            if "CE" in option:
                all_options.append({
                    'Strike': option['strikePrice'],
                    'spot': data['records']['underlyingValue'],
                    'Expiry': option['expiryDate'],
                    'IV': option['CE'].get('impliedVolatility', 0) / 100,
                    'LTP': option['CE'].get('lastPrice', 0),
                    'OI': option['CE'].get('openInterest', 0),
                    'option_type': 'call'
                })

            if "PE" in option:
                all_options.append({
                    'Strike': option['strikePrice'],
                    'spot': data['records']['underlyingValue'],
                    'Expiry': option['expiryDate'],
                    'IV': option['PE'].get('impliedVolatility', 0) / 100,
                    'LTP': option['PE'].get('lastPrice', 0),
                    'OI': option['PE'].get('openInterest', 0),
                    'option_type': 'put'
                })

        self.df = pd.DataFrame(all_options)
        self.expiries = sorted(self.df['Expiry'].unique()) if not self.df.empty else []

    def black_scholes_greeks(self, S, K, T, r, sigma, option_type='call'):
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        delta = norm.cdf(d1) if option_type == 'call' else -norm.cdf(-d1)
        gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
        vega = S * norm.pdf(d1) * np.sqrt(T) / 100
        theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))) / 365
        theta -= r * K * np.exp(-r * T) * norm.cdf(d2) / 365 if option_type == 'call' else -r * K * np.exp(-r * T) * norm.cdf(-d2) / 365

        return {'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta}

    def layout(self):
        self.app.layout = html.Div([
            html.H1('Nifty Option Chain Dashboard'),
            dcc.Dropdown(id='expiry-dropdown', options=[], placeholder='Select Expiry'),
            html.H3('Call Options'),
            dash_table.DataTable(id='calls-table', page_size=10),
            html.H3('Put Options'),
            dash_table.DataTable(id='puts-table', page_size=10),
            
            html.H3('Open Interest Chart'),
            dcc.Graph(id='oi-chart'),
            dcc.Interval(id='interval', interval=60000000, n_intervals=0)
            
        ])

    def register_callbacks(self):
        @self.app.callback(
            Output('expiry-dropdown', 'options'),
            Input('interval', 'n_intervals')
        )
        def update_expiry_dropdown(n):
            self.fetch_options_chain()
            return [{'label': exp, 'value': exp} for exp in self.expiries]

        @self.app.callback(
            [Output('calls-table', 'data'), Output('puts-table', 'data'),
             Output('oi-chart', 'figure')],
            [Input('expiry-dropdown', 'value'), Input('interval', 'n_intervals')]
        )
        def update_dashboard(selected_expiry, n):
            

            filtered_df = self.df[self.df['Expiry'] == selected_expiry] if selected_expiry else self.df

            T = 7 / 365
            r = 0.065

            
            filtered_df.loc[filtered_df["option_type"] == "call", ["delta", "gamma", "vega", "theta"]] = \
                filtered_df[filtered_df["option_type"] == "call"].apply(
                    lambda row: pd.Series(self.black_scholes_greeks(row['spot'], row['Strike'], T, r, row['IV'], 'call')),
                    axis=1
                )

            filtered_df.loc[filtered_df["option_type"] == "put", ["delta", "gamma", "vega", "theta"]] = \
                filtered_df[filtered_df["option_type"] == "put"].apply(
                    lambda row: pd.Series(self.black_scholes_greeks(row['spot'], row['Strike'], T, r, row['IV'], 'put')),
                    axis=1
                )

            
            calls = filtered_df[filtered_df['option_type'] == 'call']

            puts = filtered_df[filtered_df["option_type"] == "put"]

            
            oi_chart = px.bar(
                filtered_df, x="Strike", y="OI", color="option_type",
                title="Open Interest (Call & Put)", labels={"OI": "Open Interest"},
                barmode='group'
            )

            return calls.to_dict("records"), puts.to_dict("records"), oi_chart

    def run(self):
        self.app.run(debug=True)

if __name__ == "__main__":
    dashboard = NiftyOptionsDashboard()
    dashboard.run()
    


# run this in web : http://127.0.0.1:8050/
    


C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:56: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:60: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:56: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:60: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:56: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:60: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:56: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\karan\AppData\Local\Temp\ipykernel_5980\3899951951.py:60: RuntimeWarning:

invalid value en